In [2]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [1]:
import os
from langchain.docstore.document import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor, tool
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

In [3]:


# 환경 변수 설정 (필요에 따라 주석 해제 후 API 키 입력)
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# 1. 영화 정보가 담긴 문서 리스트 생성
docs = [
    Document(
        page_content="크리스토퍼 놀란 감독의 SF 영화. 꿈속으로 들어가 현실을 조작한다.",
        metadata={
            "title": "인셉션",
            "director": "크리스토퍼 놀란",
            "year": 2010,
            "genre": "SF",
            "rating": 8.8,
        },
    ),
    Document(
        page_content="거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.",
        metadata={
            "title": "인터스텔라",
            "director": "크리스토퍼 놀란",
            "year": 2014,
            "genre": "SF",
            "rating": 8.6,
        },
    ),
    Document(
        page_content="미국 대공황 시대를 배경으로 한 마술 대결 영화.",
        metadata={
            "title": "프레스티지",
            "director": "크리스토퍼 놀란",
            "year": 2006,
            "genre": "미스터리",
            "rating": 8.5,
        },
    ),
    Document(
        page_content="스파이더맨의 능력과 책임에 대한 이야기. 고등학생 영웅의 성장기.",
        metadata={
            "title": "스파이더맨: 홈커밍",
            "director": "존 왓츠",
            "year": 2017,
            "genre": "액션",
            "rating": 7.4,
        },
    ),
    Document(
        page_content="마블 히어로들이 팀을 이루어 지구를 구하는 이야기. 화려한 액션이 특징.",
        metadata={
            "title": "어벤져스",
            "director": "조스 웨던",
            "year": 2012,
            "genre": "액션",
            "rating": 8.1,
        },
    ),
]

# 2. 벡터 스토어 및 임베딩 모델 설정
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(docs, embeddings)
llm = ChatOpenAI(temperature=0)

# 3. 셀프 쿼리 리트리버를 위한 메타데이터 스키마 정의
metadata_field_info = [
    AttributeInfo(name="title", description="영화의 제목", type="string"),
    AttributeInfo(name="director", description="영화 감독의 이름", type="string"),
    AttributeInfo(name="year", description="영화가 개봉된 연도", type="integer"),
    AttributeInfo(name="genre", description="영화의 장르", type="string"),
    AttributeInfo(name="rating", description="IMDb 평점 (10점 만점)", type="float"),
]

# 4. 셀프 쿼리 리트리버 생성
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="영화의 줄거리와 특징에 대한 설명입니다.",
    metadata_field_info=metadata_field_info,
    verbose=True,
)


# 5. 셀프 쿼리 리트리버를 툴로 정의
@tool
def movie_retriever_tool(query: str) -> str:
    """
    사용자의 자연어 쿼리를 기반으로 영화를 검색합니다.
    감독, 개봉 연도, 장르, 평점과 같은 조건을 함께 사용하여 검색할 수 있습니다.
    예시: "크리스토퍼 놀란 감독이 2010년 이후에 만든 SF 영화"
    """
    return self_query_retriever.invoke(query)


# 6. 두 번째 툴: 인터넷 검색
internet_search = DuckDuckGoSearchRun()


@tool
def internet_search_tool(query: str) -> str:
    """
    웹에서 최신 정보를 검색하는 도구입니다. 영화 정보를 찾을 때 사용하세요.
    """
    try:
        return internet_search.invoke(query)
    except Exception as e:
        return f"인터넷 검색 중 오류 발생: {str(e)}"


# 7. 에이전트 및 에이전트 실행자 생성
tools = [movie_retriever_tool, internet_search_tool]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 영화 검색 전문가입니다. 사용자 질문에 답변하기 위해 주어진 툴을 활용하세요.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [4]:
# 8. 에이전트 실행
print("--- case 1: 복잡한 조건 검색 ---")
# "2010년 이후에 개봉한 크리스토퍼 놀란 감독의 SF 영화를 찾아줘"
result1 = agent_executor.invoke(
    {"input": "2010년 이후에 개봉한 크리스토퍼 놀란 감독의 SF 영화 알려줘"}
)
print(result1["output"])

--- case 1: 복잡한 조건 검색 ---


> Entering new AgentExecutor chain...

Invoking: `movie_retriever_tool` with `{'query': '크리스토퍼 놀란 감독이 2010년 이후에 만든 SF 영화'}`



Invoking: `movie_retriever_tool` with `{'query': '크리스토퍼 놀란 감독이 2010년 이후에 만든 SF 영화'}`




Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(id='071d8fc7-a2f6-4241-bf42-ea6ff61e119b', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.')]크리스토퍼 놀란 감독의 2010년 이후 SF 영화 중 한 편은 "인터스텔라"입니다. 이 영화는 2014년에 개봉했으며, 거대한 우주 모험을 다루며 블랙홀과 시간 여행이 주된 소재로 사용되었습니다. 평점은 8.6점입니다.

> Finished chain.
크리스토퍼 놀란 감독의 2010년 이후 SF 영화 중 한 편은 "인터스텔라"입니다. 이 영화는 2014년에 개봉했으며, 거대한 우주 모험을 다루며 블랙홀과 시간 여행이 주된 소재로 사용되었습니다. 평점은 8.6점입니다.
크리스토퍼 놀란 감독의 2010년 이후 SF 영화 중 한 편은 "인터스텔라"입니다. 이 영화는 2014년에 개봉했으며, 거대한 우주 모험을 다루며 블랙홀과 시간 여행이 주된 소재로 사용되었습니다. 평점은 8.6점입니다.

> Finished chain.
크리스토퍼 놀란 감독의 2010년 이후 SF 영화 중 한 편은 "인터스텔라"입니다. 이 영화는 2014년에 개봉했으며, 거대한 우주 모험을 다루며 블랙홀과 시간 여행이 주된 소재로 사용되었습니다. 평점은 8.6점입니다.


In [ ]:
print("\n--- case 2: 단순한 쿼리 검색 ---")
# "마블 영화 중에서 평점 8.0점 이상인 영화를 찾아줘"
result2 = agent_executor.invoke(
    {"input": "마블 영화 중에서 평점 8.0점 이상인 영화를 찾아줘"}
)
print(result2["output"])


--- case 2: 단순한 쿼리 검색 ---


> Entering new AgentExecutor chain...

Invoking: `movie_retriever_tool` with `{'query': '마블 영화 평점 8.0 이상'}`


[Document(id='8ed6237d-1e35-4833-bbf1-a8c9fb4207c5', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='230f8df8-788f-4e8e-8c76-924a43c675b8', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='fd413176-d88e-40b1-b97e-7604927ab0b7', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.'), Document(id='20944f0e-4a3a-471c-b2c7-a92304e7acab', metadata={'director': '크리스토퍼 놀란', 'genre': 'SF', 'rating': 8.6, 'title': '인터스텔라', 'year': 2014}, page_content='거대한 우주 모험을 다룬 영화. 블랙홀과 시간 여행이 주된 소재이다.')]마블 영화 중에서 평점 8.0 이상인 영화를 찾았습니다. 

인터스텔라 (

In [ ]:
print("\n--- case 2: 단순한 쿼리 검색 ---")
# "마블 영화 중에서 평점 8.0점 이상인 영화를 찾아줘"
result2 = agent_executor.invoke({"input": "아이언맨1편에 대해 알려줘"})
print(result2["output"])


--- case 2: 단순한 쿼리 검색 ---


> Entering new AgentExecutor chain...

Invoking: `movie_retriever_tool` with `{'query': '아이언맨 1편'}`


[Document(id='d02e5fea-f160-4140-998b-0f92d1cb5fa8', metadata={'director': '크리스토퍼 놀란', 'genre': '미스터리', 'rating': 8.5, 'title': '프레스티지', 'year': 2006}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.'), Document(id='c5e2a994-ae64-4e77-bd4e-f2890afc9e12', metadata={'director': '크리스토퍼 놀란', 'genre': '미스터리', 'rating': 8.5, 'title': '프레스티지', 'year': 2006}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.'), Document(id='c599fe6c-3cf5-4ecc-b589-99a21934a701', metadata={'director': '크리스토퍼 놀란', 'genre': '미스터리', 'rating': 8.5, 'title': '프레스티지', 'year': 2006}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.'), Document(id='bcbb3197-c5ed-4631-9409-f5bdfedf592c', metadata={'director': '크리스토퍼 놀란', 'genre': '미스터리', 'rating': 8.5, 'title': '프레스티지', 'year': 2006}, page_content='미국 대공황 시대를 배경으로 한 마술 대결 영화.')]
Invoking: `internet_search_tool` with `{'query': '아이언맨 1편'}`
responded: 아이언맨 1편에 대한 